# VHH SMEFT predictions — NNLO QCD

This notebook computes **linear SMEFT** predictions for $W^\pm HH$ and $ZHH$ at **LO**
and **NNLO QCD** (HHZ for $ZHH$), for the LHC at 13.6 or 14 TeV.

The cross section expands as $\sigma = \sigma_{\mathrm{SM}} + \sum_i B_i C_i$
with Wilson coefficients $C_i$ in $\mathrm{TeV}^{-2}$ and $B_i$ in $\mathrm{fb\,TeV}^2$
bundled under `data/SMEFT/`.

> **Install once** with `pip install -e ".[notebook]"` (package only), then open this
> notebook from the repo root. Restart the kernel after editing `vhh_predict/`.

> For **HEFT** $\kappa$ predictions use [`vhh_prediction_HEFT.ipynb`](vhh_prediction_HEFT.ipynb).

## Contents

| § | What it does | Main output |
|---|---|---|
| Setup | Editable install check + imports | — |
| 1 | Process, energy, output flags | `analysis` |
| 2 | Spot check at one WC point | results table (+ simulation if enabled) |
| 3 | Scan one WC axis and plot | `scan_data` + PNGs (+ optional `.txt`) |
| 4 | Joint multi-axis grid scan | one `results/points/smeft/*_x_*.txt` |
| 5 | SMEFT benchmark tables | display + `results/tables/smeft/wc_intervals.tex` |

See [README.md](README.md) for an overview; `AGENTS.md` for the package API.


## Setup

Requires an editable install from the repo root (defined in `pyproject.toml`):

```bash
pip install -e ".[notebook]"
```

Then `import vhh_predict` works — notebooks do **not** modify `sys.path`.
Restart the kernel after changing package code.


In [1]:
from vhh_predict.analysis import package_root, plots_dir, tables_dir

REPO_ROOT = package_root()
if not (REPO_ROOT / "data" / "SMEFT").is_dir() or not (REPO_ROOT / "vhh_predict").is_dir():
    raise RuntimeError(
        f"Expected 'data/SMEFT/' and 'vhh_predict/' under {REPO_ROOT}.\n"
        "Install from the repo root:  pip install -e \".[notebook]\"\n"
        "Then start Jupyter from that same directory."
    )

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from vhh_predict.plot_style import (
    default_plot_title,
    plot_style_with_layout,
    scan_plot_filename_stem,
)
from vhh_predict.plots import (
    plot_sigma_nnlo_and_enhancement_nnlo,
    plot_sigma_nnlo_and_kfactor,
)
from vhh_predict.smeft_analysis import load_smeft_analysis
from vhh_predict.smeft_core import spot_check_caption, spot_check_table
from vhh_predict.smeft_operators import (
    SMEFT_WC_INTERVALS,
    SMEFT_WC_PLAIN,
    W_SCAN_WC_KEYS,
    scan_axes,
    sm_wc_values,
)
from vhh_predict.smeft_scan_io import scan_and_save, scan_grid_and_save
from vhh_predict.smeft_tables import (
    CHANNELS,
    TABLE_ENERGIES_TEV,
    ZHH_TABLE_GROUPS,
    build_channel_tables,
    latex_wc_interval_table,
)

PLOTS_DIR = plots_dir() / "smeft"
TABLES_DIR = tables_dir() / "smeft"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/michalryczkowski/Documents/Programming/GitHub/VHH-NNLO


## 1. Configuration

| Variable | Meaning |
|----------|---------|
| `PROCESS` | `"WplusHH"`, `"WminusHH"`, or `"ZHH"` |
| `ENERGY_TEV` | `13.6` or `14.0` |
| `COMPARE_SIMULATION` | Compare to bundled SMEFT `.out` files in §2 only |
| `UNCERTAINTIES_AS_PERCENT` | Print uncertainties as % (`False` → fb) |
| `SAVE_SCAN_POINTS` | Write §3 / §4 scan tables to `results/points/smeft/` |
| `SAVE_PLOTS` | Write §3 plot PNGs to `results/plots/smeft/` |
| `SIGMA_INSET`, `LEGEND_LOC`, `INSET_LOC` | Plot layout (§3) |

Wilson coefficients $C_i$ in $\mathrm{TeV}^{-2}$. SM point: all $C_i = 0$.

**Allowed SMEFT intervals** (`SMEFT_WC_INTERVALS` in `vhh_predict/smeft_operators.py`):

| Bosonic | Interval [TeV$^{-2}$] |
|---------|----------------------|
| $C_\varphi$ | [−15, 5] |
| $C_{\varphi W}$ | [−1, 1] |
| $C_{\varphi B}$ | [−0.5, 0.5] |
| $C_{\varphi WB}$ | [−1.5, 1.5] |
| $C_{\varphi D}$ | [−2, 2] |
| $C_{\varphi\square}$ | [−1.5, 1.5] |

| Fermionic | Interval [TeV$^{-2}$] |
|-----------|----------------------|
| $C_{\varphi q}^{(3)}$ | [−0.2, 0.05] |
| $C_{\varphi t}$ | [−25, 34] |
| $C_{\varphi Q}^{(3)}$ | [−8, 2] |
| $C_{\varphi Q}^{(1)}$ | [−6.5, 30.5] |
| $C_{\varphi t}+C_{\varphi Q}^{(3)}-C_{\varphi Q}^{(1)}$ | [−8, 2] |
| $C_{\varphi q}^{(1)}$ | [−3, 1] |
| $C_{\varphi u}$ | [−3.5, 1] |
| $C_{\varphi d}$ | [−4, 4] |
| $C_{t\varphi}$ | [−15, 5] |

> **$C_{t\varphi}$ ≠ $C_{\varphi t}$:** $C_{t\varphi}$ ($B_{11}$, Fortran `cth`) is distinct from $C_{\varphi t}$ (enters the $B_{12}$ combination).


In [2]:
PROCESS = "ZHH"
ENERGY_TEV = 14.0

COMPARE_SIMULATION = True
UNCERTAINTIES_AS_PERCENT = True
SAVE_SCAN_POINTS = True
SAVE_PLOTS = True

SIGMA_INSET = False
LEGEND_LOC = "lower right"
INSET_LOC = "upper left"

PLOT_STYLE = plot_style_with_layout(
    legend_loc=LEGEND_LOC,
    inset_loc=INSET_LOC,
    sigma_inset=SIGMA_INSET,
)

analysis = load_smeft_analysis(PROCESS, ENERGY_TEV)
nn_label = analysis.nnlo_label

print(f"{PROCESS} @ {ENERGY_TEV} TeV  |  scan axes: {', '.join(scan_axes(PROCESS))}")


ZHH @ 14.0 TeV  |  scan axes: phi, phiBox, phiD, phiq3st, phiW, phiq1st, phiu, phid, phiB, phiWB


## 2. Spot check

Set **`WCS`** — a dictionary of Wilson coefficients in $\mathrm{TeV}^{-2}$.
Unlisted coefficients default to 0 (SM).

Example for $ZHH$: `{"phiW": -0.2}` scans $C_{\varphi W} = -0.2$ with all other $C_i = 0$.

When `COMPARE_SIMULATION=True`, bundled `.out` files under `data/SMEFT/.../Simulation/`
are matched by filename-encoded Wilson coefficients.


In [ ]:
WCS = {"phiq1rd": -3}  # SM: {} or all zeros

display(Markdown(f"### {spot_check_caption(analysis, WCS)}"))
display(
    spot_check_table(
        analysis,
        WCS,
        as_percent=UNCERTAINTIES_AS_PERCENT,
        compare_simulation=COMPARE_SIMULATION,
    )
)


ValueError: Unknown WC 'phiq1rd' for ZHH; allowed: ['phi', 'phiB', 'phiBox', 'phiD', 'phiQ3rd', 'phiW', 'phiWB', 'phid', 'phiq1st', 'phiq3st', 'phiu', 'tphi']

## 3. Scan and plots

Vary **one** Wilson coefficient; other components stay at `FIXED_WCS`.

- `scan_axis` — one of the keys printed in §1 (`phi`, `phiW`, …)
- `scan_vmin`, `scan_vmax` — scan window (defaults below use the global-fit interval)
- `scan_n_points` — grid resolution


In [ ]:
FIXED_WCS = {}  # SM background; e.g. {"phi": 0.0, "phiW": 0.0, ...}

scan_axis = "phiW"
scan_vmin, scan_vmax = SMEFT_WC_INTERVALS[scan_axis]
scan_n_points = 400

print(f"Interval for {SMEFT_WC_PLAIN[scan_axis]}: [{scan_vmin:g}, {scan_vmax:g}]")
print(f"Scan / plot window:                  [{scan_vmin:g}, {scan_vmax:g}]")

scan_data, scan_points_file = scan_and_save(
    analysis,
    scan_axis,
    vmin=scan_vmin,
    vmax=scan_vmax,
    fixed_wcs=FIXED_WCS,
    n_points=scan_n_points,
    uncertainties=True,
    save=SAVE_SCAN_POINTS,
)
if SAVE_SCAN_POINTS:
    print(f"Saved {scan_points_file}")

plot_title = default_plot_title(PROCESS, ENERGY_TEV, nn_label)
prefix = scan_plot_filename_stem(PROCESS, ENERGY_TEV, scan_axis, scan_vmin, scan_vmax)

_nnlo_kw = dict(
    style=PLOT_STYLE,
    nnlo_label=nn_label,
    xmin=scan_vmin,
    xmax=scan_vmax,
    save=SAVE_PLOTS,
)

plot_sigma_nnlo_and_kfactor(
    scan_data,
    title=plot_title,
    sigma_inset=SIGMA_INSET,
    output=PLOTS_DIR / f"{prefix}_sigma_nnlo_K.png",
    **_nnlo_kw,
)
plt.show()

plot_sigma_nnlo_and_enhancement_nnlo(
    scan_data,
    title=plot_title,
    sigma_inset=SIGMA_INSET,
    show_enhancement_uncertainty=False,
    output=PLOTS_DIR / f"{prefix}_sigma_nnlo_sigmaSM_{nn_label}.png",
    **_nnlo_kw,
)
plt.show()


## 4. Joint multi-axis scan (Cartesian grid)

Vary **all** listed Wilson coefficients **at once** (Cartesian product) and write **one** `.txt` under `results/points/smeft/`.

- `FIXED_WCS` — non-scanned $C_i$ (defaults SM = 0)
- `BATCH_SCAN_AXES` — axes to vary jointly
- `BATCH_SCAN_WINDOWS` — optional `{axis: (vmin, vmax)}`; omitted axes use `SMEFT_WC_INTERVALS`
- `BATCH_SCAN_N_POINTS` — points **per axis** (2 × 40 → 1600 rows)

Set `BATCH_SCAN_AXES = ()` to skip.


In [ ]:
FIXED_WCS = {}

BATCH_SCAN_AXES = (
    "phi",
    "phiW",
)
BATCH_SCAN_N_POINTS = 40  # per axis → 40×40 = 1600 points for 2 axes
BATCH_SCAN_WINDOWS = {}

if BATCH_SCAN_AXES:
    from vhh_predict.smeft_core import resolve_scan_axis

    scan_data, path = scan_grid_and_save(
        analysis,
        BATCH_SCAN_AXES,
        windows=BATCH_SCAN_WINDOWS or None,
        n_points=BATCH_SCAN_N_POINTS,
        fixed_wcs=FIXED_WCS,
        save=SAVE_SCAN_POINTS,
        uncertainties=False,
    )
    n_total = len(next(iter(scan_data.values())))
    print(f"Joint grid scan: {len(BATCH_SCAN_AXES)} axes × {BATCH_SCAN_N_POINTS} pts → {n_total} points")
    for axis in BATCH_SCAN_AXES:
        lo, hi = (BATCH_SCAN_WINDOWS or {}).get(axis) or SMEFT_WC_INTERVALS[axis]
        print(f"  {SMEFT_WC_PLAIN[axis]}  [{lo:g}, {hi:g}]")
    print(f"  →  {path if SAVE_SCAN_POINTS else '(not saved)'}")
else:
    print("Joint scan skipped (BATCH_SCAN_AXES is empty).")


## 5. SMEFT benchmark tables

**Not all Wilson coefficients** — only a fixed subset defined in the package:

| Channel | WCs in σ tables | Where defined |
|---------|-----------------|---------------|
| `WplusHH`, `WminusHH` | all LO keys: `phi`, `phiBox`, `phiD`, `phiq3st`, `phiW` | `W_SCAN_WC_KEYS` |
| `ZHH` | two groups: `(phi, phiW)` and `(phiq3st, phiD)` | `ZHH_TABLE_GROUPS` |

For each selected WC, σ is evaluated at SM and at that WC’s interval **min/max** (others held at SM = 0), for both 13.6 and 14 TeV.

The LaTeX file `results/tables/smeft/wc_intervals.tex` lists **all** allowed intervals (bosonic + fermionic), including WCs that are *not* in the σ tables above.

To change which WCs appear in the σ tables, edit `ZHH_TABLE_GROUPS` in `vhh_predict/smeft_tables.py` (ZHH) or `W_SCAN_WC_KEYS` in `vhh_predict/smeft_operators.py` (W±), then restart the kernel.


In [ ]:
latex_path = TABLES_DIR / "wc_intervals.tex"
latex_path.write_text(latex_wc_interval_table(), encoding="utf-8")
print(f"Saved interval table (all WCs): {latex_path}")

print("σ benchmark WCs:")
print(f"  W±HH: {', '.join(SMEFT_WC_PLAIN[k] for k in W_SCAN_WC_KEYS)}")
print(
    "  ZHH groups: "
    + " | ".join(
        "(" + ", ".join(SMEFT_WC_PLAIN[k] for k in group) + ")"
        for group in ZHH_TABLE_GROUPS
    )
)

for process in CHANNELS:
    display(Markdown(f"## {process}"))
    tables = build_channel_tables(process, energies_tev=TABLE_ENERGIES_TEV)
    for key, df in tables.items():
        if process == "ZHH":
            axes = key.split("_")
            title = ", ".join(SMEFT_WC_PLAIN.get(a, a) for a in axes)
            display(Markdown(f"### {title}"))
        else:
            display(Markdown("### all LO WCs (W_SCAN_WC_KEYS)"))
        display(df)
